<a href="https://colab.research.google.com/github/Oyedamola-bioinformatics/malaria_detection_cnn/blob/main/Malaria_detection_cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# FIXED Cell 2 - Official NIH + TensorFlow mirror (100% working)

import tensorflow as tf
import pathlib
import os

# Two working mirrors - tries both
urls = [
    "https://storage.googleapis.com/laurencemoroney-blog.appspot.com/cell_images.zip",  # TensorFlow official
    "https://data.lhncbc.nlm.nih.gov/public/Malaria/cell_images.zip"  # NIH official
]

data_dir = None
for url in urls:
    try:
        print(f"Trying {url[:40]}...")
        zip_path = tf.keras.utils.get_file("cell_images.zip", url, extract=True)
        # Find folder
        for p in pathlib.Path(zip_path).parent.rglob("cell_images"):
            if (p / "Parasitized").exists():
                data_dir = p
                break
        if data_dir:
            break
    except Exception as e:
        print(f"Failed, trying next... {e}")
        continue

if data_dir is None:
    raise Exception("Download failed - check internet")

os.environ['MALARIA_DATA_DIR'] = str(data_dir)

print(f"\n✅ SUCCESS!")
print(f"Parasitized: {len(list((data_dir/'Parasitized').glob('*.png')))}")
print(f"Uninfected: {len(list((data_dir/'Uninfected').glob('*.png')))}")
print("Ready for Cell 3")

Trying https://storage.googleapis.com/laurencem...
Failed, trying next... URL fetch failure on https://storage.googleapis.com/laurencemoroney-blog.appspot.com/cell_images.zip: 404 -- Not Found
Trying https://data.lhncbc.nlm.nih.gov/public/M...
353452851/353452851 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step

✅ SUCCESS!
Parasitized: 13779
Uninfected: 13779
Ready for Cell 3


In [ ]:
# Project 3: Automated Malaria Detection using CNN
# Dataset: NIH Malaria Cell Images

import tensorflow as tf
print(f"TensorFlow Version: {tf.__version__}")

import pathlib
import os

print("Environment ready.")

TensorFlow Version: 2.21.0
Environment ready.


In [ ]:
# Cell 3 - CNN Training - Professional Version
from tensorflow.keras import layers, models
import pathlib, os

data_dir = pathlib.Path(os.environ['MALARIA_DATA_DIR'])
IMG_SIZE = 128
BATCH_SIZE = 16

print(f"Loading from {data_dir}...")

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="training", seed=42,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE)
val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="validation", seed=42,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE)

print("Building CNN...")

model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.Conv2D(16, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy',
              metrics=['accuracy', 'Precision', 'Recall'])

model.summary()

print("\n🔥 TRAINING - 3 epochs (~4 mins on phone)...")
history = model.fit(train_ds, validation_data=val_ds, epochs=3)

# Save
model.save("malaria_cnn_model.h5")

print(f"\n✅ TRAINING COMPLETE!")
print(f"Training Accuracy: {history.history['accuracy'][-1]:.4f}")
print(f"Validation Accuracy: {history.history['val_accuracy'][-1]:.4f}")
print("\nPortfolio ready - Type GITHUB for README!")

Loading from /root/.keras/datasets/cell_images_extracted/cell_images...
Found 27558 files belonging to 2 classes.
Using 22047 files for training.
Found 27558 files belonging to 2 classes.
Using 5511 files for validation.
Building CNN...


/usr/local/lib/python3.13/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 126, 126, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 12544)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     1,605,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,629,473 (6.22 MB)

 Trainable params: 1,629,473 (6.22 MB)

 Non-trainable params: 0 (0.00 B)


🔥 TRAINING - 3 epochs (~4 mins on phone)...
Epoch 1/3
1378/1378 ━━━━━━━━━━━━━━━━━━━━ 385s 278ms/step - Precision: 0.8370 - Recall: 0.8296 - accuracy: 0.8346 - loss: 0.3374 - val_Precision: 0.9300 - val_Recall: 0.9699 - val_accuracy: 0.9477 - val_loss: 0.1632
Epoch 2/3
1378/1378 ━━━━━━━━━━━━━━━━━━━━ 389s 282ms/step - Precision: 0.9406 - Recall: 0.9651 - accuracy: 0.9523 - loss: 0.1490 - val_Precision: 0.9202 - val_Recall: 0.9868 - val_accuracy: 0.9499 - val_loss: 0.1534
Epoch 3/3
1378/1378 ━━━━━━━━━━━━━━━━━━━━ 372s 270ms/step - Precision: 0.9498 - Recall: 0.9709 - accuracy: 0.9599 - loss: 0.1233 - val_Precision: 0.9344 - val_Recall: 0.9839 - val_accuracy: 0.9568 - val_loss: 0.1370



✅ TRAINING COMPLETE!
Training Accuracy: 0.9599
Validation Accuracy: 0.9568

Portfolio ready - Type GITHUB for README!
